In [ ]:
!pip install mlx-embeddings-lora

In [ ]:
from pathlib import Path

import mlx.optimizers as optim

from mlx_lm.tuner.callbacks import TrainingCallback, WandBCallback

from mlx_embeddings_lora.trainer.contrastive_trainer import (
    TrainingArgs,
    evaluate,
    train,
)
from mlx_embeddings_lora.trainer.dataset import CacheDataset, ContrastiveLearningDataset
from mlx_embeddings_lora.trainer.utils import from_pretrained, fuse_and_save_model

from datasets import load_dataset


In [ ]:
model = "Qwen/Qwen3-Embedding-0.6B"

lora_config = {
    "rank": 16, # The higer the smarter but uses more resources
    "num_layers": 12,
    "dropout": 0.0,
    "scale": 10.0,
    "use_dora": False
}

quanziation_config = {"bits": 8, "group_size": 64} # Quantize the model to 8bits

dataset = "Goekdeniz-Guelmez/sentence-compression-pairs" # Dataset has to have this format Goekdeniz-Guelmez/semantic-triplets-dataset-test, the "negative" field is optional like Goekdeniz-Guelmez/sentence-compression-pairs
adapter_path = "path/to/adapter"

In [ ]:
model, tokenizer = from_pretrained(
    model=model,
    adapter_path=adapter_path,
    lora_config=lora_config,
    quantized_load=quanziation_config
)

In [ ]:
from mlx_lm.tuner.utils import print_trainable_parameters
print_trainable_parameters(model)

In [ ]:
train_set = ContrastiveLearningDataset(load_dataset(dataset)["train"], tokenizer=tokenizer)
valid_set = ContrastiveLearningDataset(load_dataset(dataset)["valid"], tokenizer=tokenizer)
test_set = ContrastiveLearningDataset(load_dataset(dataset)["test"], tokenizer=tokenizer)

In [ ]:
print(test_set[0])

In [ ]:
opt = optim.AdamW(learning_rate=1e-5)

In [ ]:
train(
    model=model,
    optimizer=opt,
    loss_fn="infonce",
    similarity="cosine", # can be cosine and sine
    train_dataset=CacheDataset(train_set),
    val_dataset=CacheDataset(valid_set),
    training_callback=TrainingCallback(), # TrainingCallback() or WandBCallback()
    args=TrainingArgs(
        batch_size=1,
        iters=100,
        val_batches=1,
        steps_per_report=10,
        steps_per_eval=500,
        steps_per_save=1000,
        adapter_file=Path(adapter_path) / "adapters.safetensors",
        temperature=0.7,
        margin=0.5,
        max_seq_length=1024,
        grad_checkpoint=True,
        gradient_accumulation_steps=2,
    )
)

In [ ]:
evaluate(
    model,
    dataset=CacheDataset(test_set),
    batch_size=1,
    num_batches=1,
    max_seq_length=1024,
    loss_fn="infonce",
    similarity="cosine",
    temperature=0.7,
    margin=0.5
)

In [ ]:
fuse_and_save_model(
    model=model,
    tokenizer=tokenizer,
    save_path="fused_model",
    de_quantize=True
)